# FAIR Toolbox WP1 Maturity Metrics

This is a Google Colab notebook fetches some research software quality indicators and combine them into a maturity metric.

The notebook first installs two third-party tools, namely the [howfairis](https://github.com/fair-software/howfairis) software FAIR compliance monitor and the [Lizard](https://github.com/terryyin/lizard) code complexity analyzer.

The notebook then uses the EuropePMC API to retrieve the number of citations and open access status of the (primary) publication associated with the tool or service.

The notebook then runs howfairis and Lizard in order.

The mapped indicator identifiers are from https://zenodo.org/records/15474784.

1. Install howfairis (if fails, run again):

In [ ]:
!pip3 install --user howfairis

import os, shutil

os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")
print(shutil.which("howfairis"))

2. Install Lizard:

In [ ]:
!pip install lizard

3. Fetch the tools master spreadsheet:

In [ ]:
import pandas as pd

# Fetch the master spreadsheet with all tools
sheet_id = "SHEET_ID"
sheet_name = "Tools"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
df = pd.read_csv(url)

# Extract a specific column, e.g., "Gene"
column_data = df["URL (GitHub/GitLab if available)"]

4. Collect citations and open access status of papers as measures of scientific impact and FAIRness of the dissemination and documentation about the software:

In [ ]:
import requests
import pandas as pd

def fetch_citations_and_oa(pmids):
    base_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    citation_counts_and_oa = []

    for pmid in pmids:
        query = f"EXT_ID:{pmid} AND SRC:MED"
        params = {
            "query": query,
            "format": "json",
        }
        response = requests.get(base_url, params=params)

        # Fetch the number of citations to paper
        if response.status_code == 200:
            data = response.json()
            try:
                result = data["resultList"]["result"][0]
                count = int(result.get("citedByCount", 0))
            except (IndexError, KeyError):
                count = None
            try:
                result = data["resultList"]["result"][0]
                is_oa = result.get("isOpenAccess", None)
                if is_oa is not None:
                    is_oa = is_oa.lower() == "y"
            except (IndexError, KeyError):
                is_oa = None
        else:
            count = None
            is_oa = None

        # Check if the paper is open access
        if response.status_code == 200:
            data = response.json()
            try:
                result = data["resultList"]["result"][0]
                is_oa = result.get("isOpenAccess", None)
                if is_oa is not None:
                    is_oa = is_oa.lower() == "y"  # Convert "Y"/"N" to True/False
            except (IndexError, KeyError):
                is_oa = None
        else:
            is_oa = None

        citation_counts_and_oa.append({"PMID": pmid, "Citations": count, "OpenAccess": is_oa})

    return pd.DataFrame(citation_counts_and_oa)

PMIDs = df["PMID"]
citations = fetch_citations_and_oa(PMIDs)
citations.to_csv("citation_counts_and_oa.csv", index=False)

5. Run howfairis on all GitHub and GitLab repos:

In [ ]:
import os
import csv
import re
import shutil
import subprocess
import time
import sys

# Find the correct path of howfairis executable
howfairis_path = shutil.which("howfairis")

if howfairis_path is None:
  print("Error: howfairis executable not found. Please ensure it is installed and in your PATH.")
else:
  print(f"Found howfairis executable at: {howfairis_path}")
  csv_file = "howfairis.csv"
  with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["URL", "Howfairis checklist", "Howfairis citation", "Howfairis license", "Howfairis registry", "Howfairis repository"])

  # Go through each GitHub repo
  pattern = r"git..b\.com" # for all github.com and gitlab.com repos
  for url in column_data.dropna():
    if re.search(pattern, url):
      print(url, "is a GitHub or GitLab repo")
      # Run the CLI command
      result = subprocess.run(
          [howfairis_path, url],
          stdout=subprocess.PIPE,
          stderr=subprocess.PIPE,
          text=True
      )
      # Search for the compliance line
      compliance_line = None
      for line in result.stdout.splitlines():
          if "Calculated compliance" in line:
              compliance_line = line.strip()
              break

      # Show the result
      print("Extracted compliance line:", compliance_line)

      # Map visual symbols to boolean
      symbol_to_bool = {'●': True, '○': False}

      # FAIR indicators in the correct order
      criteria = ["repository", "license", "registry", "citation", "checklist"]

      # Desired output order
      output_order = ["checklist", "citation", "license", "registry", "repository"]

      # Extract symbols and build dict
      if compliance_line:
          symbols = compliance_line.split(":")[-1].strip().split()
          compliance_status = dict(zip(criteria, [symbol_to_bool.get(s, False) for s in symbols]))

          # Format output in the desired order
          row_values = [url] + [f"{k}:{str(compliance_status[k]).lower()}" for k in output_order]

          with open(csv_file, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(row_values)
      else:
          print("Compliance line not found.")
          with open(csv_file, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([url, "NA", "NA", "NA", "NA", "NA"])

      # Wait one minute until next howfairis call
      time.sleep(60)

    else:
      print("Not a GitHub or GitLab repo")
      with open(csv_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([url, "NA", "NA", "NA", "NA", "NA"])

6. Clone GitHub repos and run Lizard to extract source code quality indicators:

In [ ]:
import re
import lizard
import subprocess
import csv
import os
import shutil

csv_file = "lizard_summary.csv"
with open(csv_file, mode='w', newline='') as file:
  writer = csv.writer(file)
  writer.writerow(["GitHub Repo URL", "Sc1 (total NLOC)", "Su5 (total CCN)", "Su6 (average CCN)", "Sc3 (duplicates)"])

# Go through and clone each GitHub repo
pattern = r"git..b\.com" # for all github.com and gitlab.com repos
for url in column_data.dropna(): # use for url in column_data
  if re.search(pattern, url):
    print(url, "is GitHub/GitLab repo")
    !git clone {url} myrepo
    results = lizard.analyze(["myrepo"])

    # Run Lizard with duplicate detection (-Eduplicate) on a folder or file
    duplicate_result = subprocess.run(
      ["lizard", "-Eduplicate", "myrepo"],
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
      text=True
    )
    match = re.search(r"Total duplicate rate:\s+([\d.]+)%", duplicate_result.stdout)

    if match:
      duplicate_rate = float(match.group(1))
      print(f"Duplicate rate: {duplicate_rate:.2f}%")
    else:
      print("Duplicate rate not found.")

    total_nloc = 0
    total_ccn = 0
    function_count = 0

    for file in results:
      for func in file.function_list:
        total_nloc += func.nloc
        total_ccn += func.cyclomatic_complexity
        function_count += 1

    average_ccn = total_ccn / function_count if function_count > 0 else 0

    print(f"Sc1 (total NLOC): {total_nloc}")
    print(f"Su5 (total CCN): {total_ccn}")
    print(f"Su6 (average CCN): {average_ccn:.2f}")
    print(f"Sc3 (duplicates): {duplicate_rate:.2f}%")

    with open(csv_file, mode='a', newline='') as file:
      writer = csv.writer(file)
      writer.writerow([url, total_nloc, total_ccn, round(average_ccn, 2), round(duplicate_rate, 2)])

    if os.path.exists("myrepo"):
      shutil.rmtree("myrepo")
  else:
    with open(csv_file, mode='a', newline='') as file:
      writer = csv.writer(file)
      writer.writerow([url, "NA", "NA", "NA", "NA"])

7. Install PyGithub:

In [ ]:
!pip install PyGithub

8. Extract additional metrics from GitHub/GitLab JSON:

In [ ]:
from github import Github
import csv
import re
from statistics import mean
from datetime import datetime, timezone, timedelta
from urllib.parse import urlparse
import json

def to_owner_repo(s: str) -> str:
    s = s.strip()
    if s.startswith(("http://", "https://")):
        p = urlparse(s)
        parts = [x for x in p.path.strip("/").split("/") if x]
        return "/".join(parts[:2])
    if s.startswith("git@github.com:"):
        return "/".join(s.split(":",1)[1].rstrip("/").removesuffix(".git").split("/")[:2])
    return "/".join(s.strip("/").removesuffix(".git").split("/")[:2])

def repo_health(repo, token, closed_window=300):
    gh = Github(token)
    r = gh.get_repo(f"{repo}")

    # counts (PyGithub separates issues vs PRs cleanly)
    open_issues = r.get_issues(state="open").totalCount
    branches = list(r.get_branches())  # paginate internally
    protected = sum(b.protected for b in branches)
    forks = r.forks_count

    # languages
    languages = r.get_languages()  # dict: {"Python": 12345, ...}

    branch = r.get_branch(r.default_branch)
    last_commit = r.get_commit(branch.commit.sha)
    last_commit_date = last_commit.commit.author.date

    contributors = [
        {"login": c.login, "contributions": c.contributions}
        for c in r.get_contributors()
    ]

    # closed issues window for avg time-to-close
    closed = r.get_issues(state="closed")
    secs = []
    for i, it in enumerate(closed):
        if i >= closed_window: break
        if it.pull_request is not None:  # skip PRs
            continue
        if it.created_at and it.closed_at:
            secs.append((it.closed_at - it.created_at).total_seconds())

    return {
        "repo": f"{repo}",
        "default_branch": r.default_branch,
        "forks": forks,
        "open_issues": open_issues,
        "avg_time_to_close_days": round(mean(secs) / 86400, 3) if secs else None,
        "last_commit_date": last_commit.commit.author.date.strftime("%Y-%m-%d"),
        "branches_total": len(branches),
        "branches_protected": protected,
        "default_branch_is_protected": next(
            (b.protected for b in branches if b.name == r.default_branch), None),
        "languages": [lang for lang, _ in sorted(languages.items(), key=lambda x: x[1], reverse=True)],
        "contributors": [
          {"login": c.login, "total_commits": c.contributions}
          for c in sorted(r.get_contributors(), key=lambda c: c.contributions, reverse=True)
          if c.login is not None
        ],
    }

csv_file = "repo_summary.csv"
with open(csv_file, mode='w', newline='') as file:
  writer = csv.writer(file)
  writer.writerow(["Repo URL", "Default branch", "Com8 (forks)", "Is1 (open issues)", "Is2 (average time to close issue)",
                   "XxN (last commit date)", "Co3 (total branches)", "Co3 (protected branches)", "Co13 (default branch protected)",
                   "XxN (languages)","XxN (contributors)"])

  # Go through and clone each GitHub repo
  pattern = r"github\.com" # for only  github.com repos
  # pattern = r"compareMS2" # for only this tool
  for url in column_data.dropna(): # use for url in column_data
    if re.search(pattern, url):
      print(url, "is GitHub repo")
      try:
        repo=to_owner_repo(url)
        d = repo_health(repo=repo, token="GITHUB_TOKEN")
        row = d.copy()

        # serialize complex fields so the CSV remains unambiguous
        row["languages"] = json.dumps(d["languages"], ensure_ascii=False)
        row["contributors"] = json.dumps(d["contributors"], ensure_ascii=False)
        writer.writerow(row.values()) # Use .values() to write the dictionary values

      except Exception as e:
        print(f"Error processing {url}: {e}")
        # write a partial row with error info (optional)
        writer.writerow([
          url, # Write the original url
          "NA",
          "NA",
          "NA",
          "NA",
          "NA",
          "NA",
          "NA",
          "NA",
          "NA",
          "NA",
        ])
    else:
      writer.writerow([url, "NA", "NA", "NA", "NA", "NA", "NA", "NA", "NA", "NA", "NA",
        ])

9. Calculate some derived metrics:

In [ ]:
import pandas as pd
import json
import math

def safe_json_load(val):
    """Safely parse a JSON-like string to a Python object."""
    if pd.isna(val):
        return []
    val_str = str(val).strip()
    if val_str in ("", "[]", "NA"):
        return []
    try:
        return json.loads(val_str)
    except json.JSONDecodeError:
        return []

def calc_metrics(commit_list):
    if not commit_list:  # empty or no contributors
        return pd.Series({
            "HHI": None,
            "BusFactor": None,
            "Shannon": None,
            "InverseSimpson": None
        })

    total = sum(d.get("total_commits", 0) for d in commit_list)
    if total == 0:
        return pd.Series({
            "HHI": None,
            "BusFactor": None,
            "Shannon": None,
            "InverseSimpson": None
        })

    shares = [d["total_commits"] / total for d in commit_list if d.get("total_commits", 0) > 0]

    # HHI
    hhi = sum(s**2 for s in shares)

    # Inverse Simpson index
    inv_simpson = 1 / hhi if hhi != 0 else None

    # Shannon entropy
    shannon = -sum(s * math.log(s, 2) for s in shares)

    # Bus factor
    sorted_shares = sorted(shares, reverse=True)
    cum_share = 0
    bus_factor = 0
    for s in sorted_shares:
        cum_share += s
        bus_factor += 1
        if cum_share > 0.5:
            break

    return pd.Series({
        "HHI": hhi,
        "BusFactor": bus_factor,
        "Shannon": shannon,
        "InverseSimpson": inv_simpson
    })

# Parse the column from CSV
df["XxN (contributors)"] = df["XxN (contributors)"].apply(safe_json_load)

# Apply metrics
df[["HHI", "BusFactor", "Shannon", "InverseSimpson"]] = df["XxN (contributors)"].apply(calc_metrics)

print(df[["HHI", "BusFactor", "Shannon", "InverseSimpson"]])

df.to_csv("commit_metrics_output.csv", index=False)